In [0]:
data = [(1, "Ravi", "Hyderabad", 25), (2, None, "Chennai", 32), (None, "Arun", "Hyderabad", 28), (4, "Meena", None, 30), (4, "Meena", None, 30), (5, "John", "Bangalore", -5)]

columns = ["customer_id", "name", "city", "age"]

df = spark.createDataFrame(data, columns)
df.show()


+-----------+-----+---------+---+
|customer_id| name|     city|age|
+-----------+-----+---------+---+
|          1| Ravi|Hyderabad| 25|
|          2| NULL|  Chennai| 32|
|       NULL| Arun|Hyderabad| 28|
|          4|Meena|     NULL| 30|
|          4|Meena|     NULL| 30|
|          5| John|Bangalore| -5|
+-----------+-----+---------+---+



##Check Schema

In [0]:
df.printSchema()
print("Rows before cleaning:", df.count())

root
 |-- customer_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: long (nullable = true)

Rows before cleaning: 6


## find null values

In [0]:
from pyspark.sql.functions import col

df.filter(col("customer_id").isNull() | col("name").isNull() | col("city").isNull()).show()

+-----------+-----+---------+---+
|customer_id| name|     city|age|
+-----------+-----+---------+---+
|          2| NULL|  Chennai| 32|
|       NULL| Arun|Hyderabad| 28|
|          4|Meena|     NULL| 30|
|          4|Meena|     NULL| 30|
+-----------+-----+---------+---+



##Remove Null customer_id

In [0]:
df1 = df.dropna(subset=["customer_id"])

df1.show()

+-----------+-----+---------+---+
|customer_id| name|     city|age|
+-----------+-----+---------+---+
|          1| Ravi|Hyderabad| 25|
|          2| NULL|  Chennai| 32|
|          4|Meena|     NULL| 30|
|          4|Meena|     NULL| 30|
|          5| John|Bangalore| -5|
+-----------+-----+---------+---+



##Fill Missing City Values

In [0]:
df2 = df1.fillna({"city": "Unknown"})

df2.show()

+-----------+-----+---------+---+
|customer_id| name|     city|age|
+-----------+-----+---------+---+
|          1| Ravi|Hyderabad| 25|
|          2| NULL|  Chennai| 32|
|          4|Meena|  Unknown| 30|
|          4|Meena|  Unknown| 30|
|          5| John|Bangalore| -5|
+-----------+-----+---------+---+



##Remove Duplicate Rows

In [0]:
df3 = df2.dropDuplicates()

df3.show()

+-----------+-----+---------+---+
|customer_id| name|     city|age|
+-----------+-----+---------+---+
|          1| Ravi|Hyderabad| 25|
|          2| NULL|  Chennai| 32|
|          4|Meena|  Unknown| 30|
|          5| John|Bangalore| -5|
+-----------+-----+---------+---+



##Remove Invalid Ages

In [0]:
df4 = df3.filter(df3.age > 0)

df4.show()

+-----------+-----+---------+---+
|customer_id| name|     city|age|
+-----------+-----+---------+---+
|          1| Ravi|Hyderabad| 25|
|          2| NULL|  Chennai| 32|
|          4|Meena|  Unknown| 30|
+-----------+-----+---------+---+



##Count Rows After Cleaning

In [0]:
print("Rows after cleaning:", df4.count())

Rows after cleaning: 3


##Group By City

In [0]:
df4.groupBy("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Hyderabad|    1|
|  Chennai|    1|
|  Unknown|    1|
+---------+-----+



##Select Specific Columns

In [0]:
df4.select("name", "city").show()

+-----+---------+
| name|     city|
+-----+---------+
| Ravi|Hyderabad|
| NULL|  Chennai|
|Meena|  Unknown|
+-----+---------+



##Filter Hyderabad Customers

In [0]:
df4.filter(df4.city == "Hyderabad").show()

+-----------+----+---------+---+
|customer_id|name|     city|age|
+-----------+----+---------+---+
|          1|Ravi|Hyderabad| 25|
+-----------+----+---------+---+



##Create SQL Temporary View

In [0]:
df4.createOrReplaceTempView("customers")

##Run SQL Query

In [0]:
%sql

SELECT city, COUNT(*) AS total_customers
FROM customers
GROUP BY city

##Full ETL Pipeline in One Cell

In [0]:
clean_df = (df.dropna(subset=["customer_id"]).fillna({"city": "Unknown"}).dropDuplicates().filter("age > 0"))

clean_df.show()

+-----------+-----+---------+---+
|customer_id| name|     city|age|
+-----------+-----+---------+---+
|          1| Ravi|Hyderabad| 25|
|          2| NULL|  Chennai| 32|
|          4|Meena|  Unknown| 30|
+-----------+-----+---------+---+



##Read CSV File


In [0]:
csv_df = spark.read.option("header", "true").csv("/Volumes/workspace/default/session1-2027/empData.csv")

csv_df.show()

+------+------+------+-------+
|emp_id|  name|salary|address|
+------+------+------+-------+
|     1|manish| 10000|  india|
|     2|  rani| 20000|    usa|
|     3| rinku| 55000|     uk|
|     4|  neha| 12000|    usa|
|     5| sneha| 20000|     uk|
|     6| rahul| 30000|  india|
+------+------+------+-------+



##Check Schema


In [0]:
csv_df.printSchema()

root
 |-- emp_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- salary: string (nullable = true)
 |-- address: string (nullable = true)



In [0]:
csv_df.count()
csv_df.show(5)

+------+------+------+-------+
|emp_id|  name|salary|address|
+------+------+------+-------+
|     1|manish| 10000|  india|
|     2|  rani| 20000|    usa|
|     3| rinku| 55000|     uk|
|     4|  neha| 12000|    usa|
|     5| sneha| 20000|     uk|
+------+------+------+-------+
only showing top 5 rows


##Check Null Values

In [0]:
from pyspark.sql.functions import col

csv_df.filter(
    col(csv_df.columns[0]).isNull()
).show()

+------+----+------+-------+
|emp_id|name|salary|address|
+------+----+------+-------+
+------+----+------+-------+



##Remove Null Rows

In [0]:
clean_df = csv_df.dropna()

clean_df.show()

+------+------+------+-------+
|emp_id|  name|salary|address|
+------+------+------+-------+
|     1|manish| 10000|  india|
|     2|  rani| 20000|    usa|
|     3| rinku| 55000|     uk|
|     4|  neha| 12000|    usa|
|     5| sneha| 20000|     uk|
|     6| rahul| 30000|  india|
+------+------+------+-------+



##Remove Duplicate Rows

In [0]:
clean_df = clean_df.dropDuplicates()

clean_df.show()


+------+------+------+-------+
|emp_id|  name|salary|address|
+------+------+------+-------+
|     4|  neha| 12000|    usa|
|     5| sneha| 20000|     uk|
|     3| rinku| 55000|     uk|
|     6| rahul| 30000|  india|
|     1|manish| 10000|  india|
|     2|  rani| 20000|    usa|
+------+------+------+-------+



##Convert salary to Integer

In [0]:
from pyspark.sql.functions import col

csv_df = csv_df.withColumn("salary", col("salary").cast("int"))
csv_df.printSchema()

root
 |-- emp_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- address: string (nullable = true)



In [0]:
csv_df.groupBy("address").sum("salary").show()

+-------+-----------+
|address|sum(salary)|
+-------+-----------+
|  india|      40000|
|    usa|      32000|
|     uk|      75000|
+-------+-----------+

